In [ ]:
# Core libs for file handling, arrays, dataframes, and plotting
import os  # filesystem paths and directory utilities
import numpy as np  # vectorized numerical ops
import pandas as pd  # tabular data handling
import seaborn as sns  # statistical plotting
sns.set_style('darkgrid')  # consistent dark grid background for plots
import matplotlib.pyplot as plt  # plotting primitives
from sklearn.model_selection import train_test_split  # dataset splitting helper
from sklearn.metrics import confusion_matrix, classification_report  # evaluation metrics

from PIL import Image  # image I/O and basic manipulation

# Deep learning stack (TensorFlow/Keras) and computer vision helpers
import tensorflow as tf  # core TensorFlow engine
from tensorflow import keras  # Keras high-level API
from tensorflow.keras.models import Sequential  # linear stack of layers
from tensorflow.keras.optimizers import Adam, Adamax  # optimizers
from tensorflow.keras.metrics import categorical_crossentropy  # loss metric alias
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # augmentation pipeline
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization  # layer building blocks
from tensorflow.keras import regularizers  # weight regularization
import cv2  # OpenCV for image processing
import warnings  # control warnings
from tqdm import tqdm  # progress bars

from skimage.measure import shannon_entropy

from collections import Counter

from sklearn.decomposition import PCA

# Silence noisy warnings to keep notebook output clean
warnings.filterwarnings("ignore")

print('modules loaded')  # quick sanity check that imports succeeded

In [ ]:
# Define base dataset folder and split-specific paths
dataset_path = "dataset"  # root folder holding train/val/test

train_path = os.path.join(dataset_path, "train")  # training images
val_path = os.path.join(dataset_path, "val")  # validation images
test_path = os.path.join(dataset_path, "test")  # test images

classes = ["colon_aca", "colon_n"]

In [ ]:
data = []

for cls in classes:
    path = os.path.join(dataset_path, cls)
    count = len(os.listdir(path))
    data.append([cls, count])

df = pd.DataFrame(data, columns=["Class","Count"])

plt.figure(figsize=(6,4))
sns.barplot(data=df, x="Class", y="Count")
plt.title("Class Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8,4))

for i, cls in enumerate(classes):
    folder = os.path.join(dataset_path, cls)
    img_path = os.path.join(folder, os.listdir(folder)[0])
    img = Image.open(img_path)

    plt.subplot(1,2,i+1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis("off")

plt.suptitle("Sample Images")
plt.show()

In [ ]:
width = []
height = []

for cls in classes:
    folder = os.path.join(dataset_path, cls)

    for img_name in os.listdir(folder)[:300]:
        img = Image.open(os.path.join(folder,img_name))
        w,h = img.size
        width.append(w)
        height.append(h)

df = pd.DataFrame({"width":width,"height":height})

plt.figure(figsize=(6,4))
sns.scatterplot(x=df["width"], y=df["height"])
plt.title("Image Resolution Distribution")
plt.show()

In [ ]:
r,g,b = [],[],[]

for cls in classes:
    folder = os.path.join(dataset_path, cls)

    for img_name in os.listdir(folder)[:200]:
        img = cv2.imread(os.path.join(folder,img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        r.append(np.mean(img[:,:,0]))
        g.append(np.mean(img[:,:,1]))
        b.append(np.mean(img[:,:,2]))

plt.figure(figsize=(6,4))

sns.kdeplot(r,label="Red")
sns.kdeplot(g,label="Green")
sns.kdeplot(b,label="Blue")

plt.title("RGB Intensity Distribution")
plt.legend()
plt.show()

In [ ]:
brightness = []

for cls in classes:
    folder = os.path.join(dataset_path, cls)

    for img_name in os.listdir(folder)[:200]:
        img = cv2.imread(os.path.join(folder,img_name))
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        brightness.append(np.mean(hsv[:,:,2]))

plt.figure(figsize=(6,4))
sns.histplot(brightness,bins=30)

plt.title("Brightness Distribution")
plt.show()

In [ ]:
edges = []

for cls in classes:
    folder = os.path.join(dataset_path, cls)

    for img_name in os.listdir(folder)[:200]:
        img = cv2.imread(os.path.join(folder,img_name),0)
        edge = cv2.Canny(img,100,200)
        edges.append(np.mean(edge))

plt.figure(figsize=(6,4))
sns.histplot(edges,bins=30)

plt.title("Edge Strength Distribution")
plt.show()

In [ ]:
values = []
labels = []


for cls in classes:
    folder = os.path.join(dataset_path, cls)

    for img_name in os.listdir(folder)[:200]:
        img = cv2.imread(os.path.join(folder,img_name))
        values.append(np.mean(img))
        labels.append(cls)

df = pd.DataFrame({"Intensity":values,"Class":labels})

plt.figure(figsize=(6,4))
sns.boxplot(data=df,x="Class",y="Intensity")

plt.title("Pixel Intensity Comparison")
plt.show()

In [ ]:
img_path = os.path.join(dataset_path,"colon_aca",os.listdir(os.path.join(dataset_path,"colon_aca"))[0])

img = cv2.imread(img_path,0)

plt.hist(img.ravel(), bins=256)
plt.title("Grayscale Pixel Histogram")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
img_path = os.path.join("dataset","colon_aca",os.listdir("dataset/colon_aca")[0])

img = cv2.imread(img_path)

colors = ("b","g","r")

for i,color in enumerate(colors):
    hist = cv2.calcHist([img],[i],None,[256],[0,256])
    plt.plot(hist,color=color)

plt.title("RGB Color Histogram")
plt.show()

In [ ]:
def average_image(folder):

    imgs=[]

    for img_name in os.listdir(folder)[:100]:
        img=cv2.imread(os.path.join(folder,img_name))
        img=cv2.resize(img,(128,128))
        imgs.append(img)

    return np.mean(imgs,axis=0).astype("uint8")

aca_avg = average_image("dataset/colon_aca")
normal_avg = average_image("dataset/colon_n")

plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(aca_avg,cv2.COLOR_BGR2RGB))
plt.title("Average Cancer Image")

plt.subplot(1,2,2)
plt.imshow(cv2.cvtColor(normal_avg,cv2.COLOR_BGR2RGB))
plt.title("Average Normal Image")

plt.show()

In [ ]:
sizes=[]

for cls in ["colon_aca","colon_n"]:
    folder=os.path.join("dataset",cls)

    for img_name in os.listdir(folder)[:200]:
        img=Image.open(os.path.join(folder,img_name))
        sizes.append(img.size[0]*img.size[1])

plt.hist(sizes,bins=30)

plt.title("Image Size Distribution")
plt.show()

In [ ]:
contrast=[]

for cls in ["colon_aca","colon_n"]:
    folder=os.path.join("dataset",cls)

    for img_name in os.listdir(folder)[:200]:
        img=cv2.imread(os.path.join(folder,img_name),0)
        contrast.append(img.std())

sns.histplot(contrast,bins=30)

plt.title("Contrast Distribution")
plt.show()

In [ ]:
img_path=os.path.join("dataset","colon_aca",os.listdir("dataset/colon_aca")[0])

img=cv2.imread(img_path,0)

edges=cv2.Canny(img,100,200)

plt.subplot(1,2,1)
plt.imshow(img,cmap="gray")
plt.title("Original")

plt.subplot(1,2,2)
plt.imshow(edges,cmap="gray")
plt.title("Edges")

plt.show()

In [ ]:
sat=[]

for cls in ["colon_aca","colon_n"]:
    folder=os.path.join("dataset",cls)

    for img_name in os.listdir(folder)[:200]:
        img=cv2.imread(os.path.join(folder,img_name))
        hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)

        sat.append(np.mean(hsv[:,:,1]))

sns.histplot(sat,bins=30)

plt.title("Saturation Distribution")
plt.show()

In [ ]:
from skimage.measure import shannon_entropy

entropy=[]

for cls in ["colon_aca","colon_n"]:
    folder=os.path.join("dataset",cls)

    for img_name in os.listdir(folder)[:200]:
        img=cv2.imread(os.path.join(folder,img_name),0)
        entropy.append(shannon_entropy(img))

sns.histplot(entropy,bins=30)

plt.title("Image Entropy")
plt.show()

In [ ]:
res=[]

for cls in ["colon_aca","colon_n"]:
    folder=os.path.join("dataset",cls)

    for img_name in os.listdir(folder)[:200]:
        img=Image.open(os.path.join(folder,img_name))
        res.append(img.size)

Counter(res)